# 10. Retail Analytics Database with SQLite and SQL

This notebook transitions RetailPulse into an embedded SQLite database (`retailpulse.db`) to perform SQL-based analytical querying.

We evaluate key business dimensions—overall performance, monthly sales velocity, top products, geographic revenue distribution, and customer retention metrics—using SQLite queries executed via Python.

In [1]:
import sqlite3
import pandas as pd
from pathlib import Path

# Resolve project root path dynamically
current_dir = Path.cwd()
project_root = current_dir if (current_dir / "data").exists() else current_dir.parent

csv_file = project_root / "data" / "processed" / "online_retail_clean.csv"
database_file = project_root / "retailpulse.db"

# Load cleaned source data
retail = pd.read_csv(csv_file, parse_dates=["InvoiceDate"])

# Create and populate SQLite database
with sqlite3.connect(database_file) as connection:
    retail.to_sql(
        "online_retail",
        connection,
        if_exists="replace",
        index=False
    )

print(f"Database created: {database_file}")
print(f"Rows imported: {len(retail):,}")

# Verify schema and imported row count
with sqlite3.connect(database_file) as connection:
    table_info = pd.read_sql_query("PRAGMA table_info(online_retail);", connection)
    row_count = pd.read_sql_query("SELECT COUNT(*) AS total_rows FROM online_retail;", connection)

display(table_info)
display(row_count)

Database created: c:\Users\saura\OneDrive\Documents\Data Science Projects\RetailPulse\retailpulse.db
Rows imported: 536,641


,cid,name,type,notnull,dflt_value,pk
0,0,InvoiceNo,TEXT,0,None,0
1,1,StockCode,TEXT,0,None,0
2,2,Description,TEXT,0,None,0
3,3,Quantity,INTEGER,0,None,0
4,4,InvoiceDate,TIMESTAMP,0,None,0
5,5,UnitPrice,REAL,0,None,0
6,6,CustomerID,REAL,0,None,0
7,7,Country,TEXT,0,None,0
8,8,Revenue,REAL,0,None,0
9,9,IsCancellationInvoice,INTEGER,0,None,0


,total_rows
0,536641


In [2]:
# Execute comprehensive SQL queries
queries = {
    "business_overview": """
        SELECT
            COUNT(*) AS SalesLines,
            ROUND(SUM(Quantity * UnitPrice), 2) AS GrossSalesRevenue,
            SUM(Quantity) AS UnitsSold,
            COUNT(DISTINCT InvoiceNo) AS InvoiceCount,
            COUNT(DISTINCT CustomerID) AS IdentifiedCustomers
        FROM online_retail
        WHERE Quantity > 0 AND UnitPrice > 0;
    """,
    "monthly_sales": """
        SELECT
            strftime('%Y-%m', InvoiceDate) AS YearMonth,
            ROUND(SUM(Quantity * UnitPrice), 2) AS GrossRevenue,
            COUNT(DISTINCT InvoiceNo) AS InvoiceCount,
            SUM(Quantity) AS UnitsSold
        FROM online_retail
        WHERE Quantity > 0 AND UnitPrice > 0
        GROUP BY strftime('%Y-%m', InvoiceDate)
        ORDER BY YearMonth;
    """,
    "top_products": """
        SELECT
            StockCode,
            MAX(Description) AS ProductDescription,
            SUM(Quantity) AS UnitsSold,
            ROUND(SUM(Quantity * UnitPrice), 2) AS GrossRevenue,
            COUNT(DISTINCT InvoiceNo) AS InvoiceCount
        FROM online_retail
        WHERE Quantity > 0 AND UnitPrice > 0
        GROUP BY StockCode
        ORDER BY GrossRevenue DESC
        LIMIT 10;
    """,
    "country_sales": """
        SELECT
            Country,
            ROUND(SUM(Quantity * UnitPrice), 2) AS GrossRevenue,
            SUM(Quantity) AS UnitsSold,
            COUNT(DISTINCT InvoiceNo) AS InvoiceCount,
            COUNT(DISTINCT CustomerID) AS IdentifiedCustomers
        FROM online_retail
        WHERE Quantity > 0 AND UnitPrice > 0
        GROUP BY Country
        ORDER BY GrossRevenue DESC;
    """,
    "top_customers": """
        SELECT
            CustomerID,
            ROUND(SUM(Quantity * UnitPrice), 2) AS TotalRevenue,
            SUM(Quantity) AS UnitsPurchased,
            COUNT(DISTINCT InvoiceNo) AS TotalOrders
        FROM online_retail
        WHERE Quantity > 0 AND UnitPrice > 0 AND CustomerID IS NOT NULL
        GROUP BY CustomerID
        ORDER BY TotalRevenue DESC
        LIMIT 10;
    """,
    "customer_types": """
        WITH CustomerOrders AS (
            SELECT
                CustomerID,
                COUNT(DISTINCT InvoiceNo) AS TotalOrders
            FROM online_retail
            WHERE Quantity > 0 AND UnitPrice > 0 AND CustomerID IS NOT NULL
            GROUP BY CustomerID
        )
        SELECT
            CASE
                WHEN TotalOrders >= 2 THEN 'Repeat Customer'
                ELSE 'One-time Customer'
            END AS CustomerType,
            COUNT(*) AS CustomerCount
        FROM CustomerOrders
        GROUP BY CustomerType
        ORDER BY CustomerCount DESC;
    """
}

with sqlite3.connect(database_file) as connection:
    sql_results = {
        name: pd.read_sql_query(query, connection)
        for name, query in queries.items()
    }

# Display query outputs
print("=== Business Overview ===")
display(sql_results["business_overview"])

print("\n=== Monthly Trends (Head) ===")
display(sql_results["monthly_sales"].head())

print("\n=== Top 10 Products ===")
display(sql_results["top_products"])

print("\n=== Country Revenue (Top 10) ===")
display(sql_results["country_sales"].head(10))

print("\n=== Customer Order Classification ===")
display(sql_results["customer_types"])

=== Business Overview ===


,SalesLines,GrossSalesRevenue,UnitsSold,InvoiceCount,IdentifiedCustomers
0,524878,10642110.8,5572420,19960,4338



=== Monthly Trends (Head) ===


,YearMonth,GrossRevenue,InvoiceCount,UnitsSold
0,2010-12,821452.73,1559,358019
1,2011-01,689811.61,1086,387099
2,2011-02,522545.56,1100,282934
3,2011-03,716215.26,1454,376599
4,2011-04,536968.49,1246,307953



=== Top 10 Products ===


,StockCode,ProductDescription,UnitsSold,GrossRevenue,InvoiceCount
0,DOT,DOTCOM POSTAGE,706,206248.77,706
1,22423,REGENCY CAKESTAND 3 TIER,13851,174156.54,1988
2,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,168469.60,1
3,85123A,WHITE HANGING HEART T-LIGHT HOLDER,37641,104462.75,2198
4,47566,PARTY BUNTING,18283,99445.23,1685
5,85099B,JUMBO BAG RED RETROSPOT,48371,94159.81,2089
6,23166,MEDIUM CERAMIC TOP STORAGE JAR,78033,81700.92,247
7,POST,POSTAGE,3150,78101.88,1126
8,M,Manual,6984,77750.27,289
9,23084,RABBIT NIGHT LIGHT,30739,66870.03,994



=== Country Revenue (Top 10) ===


,Country,GrossRevenue,UnitsSold,InvoiceCount,IdentifiedCustomers
0,United Kingdom,9001744.09,4646906,18019,3920
1,Netherlands,285446.34,200361,94,9
2,EIRE,283140.52,147007,288,3
3,Germany,228678.40,119154,457,94
4,France,209625.37,112060,392,87
5,Australia,138453.81,83891,57,9
6,Spain,61558.56,27933,90,30
7,Switzerland,57067.60,30617,54,21
8,Belgium,41196.34,23237,98,25
9,Sweden,38367.83,36078,36,8



=== Customer Order Classification ===


,CustomerType,CustomerCount
0,Repeat Customer,2845
1,One-time Customer,1493


In [3]:
# Save CSV reports to /reports
reports_dir = project_root / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)

for report_name, report_df in sql_results.items():
    output_file = reports_dir / f"sql_{report_name}.csv"
    report_df.to_csv(output_file, index=False)
    print(f"Saved report: {output_file.name}")

# Generate standalone SQL file in /sql directory
sql_folder = project_root / "sql"
sql_folder.mkdir(parents=True, exist_ok=True)

sql_file_content = """-- RetailPulse Analytics Queries

-- 1. Business Overview
SELECT
    COUNT(*) AS SalesLines,
    ROUND(SUM(Quantity * UnitPrice), 2) AS GrossSalesRevenue,
    SUM(Quantity) AS UnitsSold,
    COUNT(DISTINCT InvoiceNo) AS InvoiceCount,
    COUNT(DISTINCT CustomerID) AS IdentifiedCustomers
FROM online_retail
WHERE Quantity > 0 AND UnitPrice > 0;

-- 2. Monthly Revenue Trends
SELECT
    strftime('%Y-%m', InvoiceDate) AS YearMonth,
    ROUND(SUM(Quantity * UnitPrice), 2) AS GrossRevenue,
    COUNT(DISTINCT InvoiceNo) AS InvoiceCount,
    SUM(Quantity) AS UnitsSold
FROM online_retail
WHERE Quantity > 0 AND UnitPrice > 0
GROUP BY strftime('%Y-%m', InvoiceDate)
ORDER BY YearMonth;

-- 3. Top 10 Products
SELECT
    StockCode,
    MAX(Description) AS ProductDescription,
    SUM(Quantity) AS UnitsSold,
    ROUND(SUM(Quantity * UnitPrice), 2) AS GrossRevenue,
    COUNT(DISTINCT InvoiceNo) AS InvoiceCount
FROM online_retail
WHERE Quantity > 0 AND UnitPrice > 0
GROUP BY StockCode
ORDER BY GrossRevenue DESC
LIMIT 10;

-- 4. Country Performance
SELECT
    Country,
    ROUND(SUM(Quantity * UnitPrice), 2) AS GrossRevenue,
    SUM(Quantity) AS UnitsSold,
    COUNT(DISTINCT InvoiceNo) AS InvoiceCount,
    COUNT(DISTINCT CustomerID) AS IdentifiedCustomers
FROM online_retail
WHERE Quantity > 0 AND UnitPrice > 0
GROUP BY Country
ORDER BY GrossRevenue DESC;

-- 5. Top 10 Customers
SELECT
    CustomerID,
    ROUND(SUM(Quantity * UnitPrice), 2) AS TotalRevenue,
    SUM(Quantity) AS UnitsPurchased,
    COUNT(DISTINCT InvoiceNo) AS TotalOrders
FROM online_retail
WHERE Quantity > 0 AND UnitPrice > 0 AND CustomerID IS NOT NULL
GROUP BY CustomerID
ORDER BY TotalRevenue DESC
LIMIT 10;

-- 6. Repeat vs One-Time Customers (CTE)
WITH CustomerOrders AS (
    SELECT
        CustomerID,
        COUNT(DISTINCT InvoiceNo) AS TotalOrders
    FROM online_retail
    WHERE Quantity > 0 AND UnitPrice > 0 AND CustomerID IS NOT NULL
    GROUP BY CustomerID
)
SELECT
    CASE
        WHEN TotalOrders >= 2 THEN 'Repeat Customer'
        ELSE 'One-time Customer'
    END AS CustomerType,
    COUNT(*) AS CustomerCount
FROM CustomerOrders
GROUP BY CustomerType
ORDER BY CustomerCount DESC;
"""

with open(sql_folder / "02_retail_analytics.sql", "w") as f:
    f.write(sql_file_content)

print(f"Saved SQL script: {sql_folder / '02_retail_analytics.sql'}")

Saved report: sql_business_overview.csv
Saved report: sql_monthly_sales.csv
Saved report: sql_top_products.csv
Saved report: sql_country_sales.csv
Saved report: sql_top_customers.csv
Saved report: sql_customer_types.csv
Saved SQL script: c:\Users\saura\OneDrive\Documents\Data Science Projects\RetailPulse\sql\02_retail_analytics.sql
